# Модель, в которой только признаки, которые можно показать клиенту
Принцип: каждый признак модели относится к группе причин, которую допустимо назвать клиенту.
Тогда показываемая причина отказа совпадает с настоящей. Проверяем, сколько это стоит в AUC.

Нужно: обновлённый src/explain.py (новые группы), models/lgbm_dev.joblib из 07.
Всё на dev, hold-out не трогаем.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

sys.path.insert(0, "..")
from src.data import load_features, split_dev_holdout
from src.explain import (EXCLUDED_GROUPS, GROUPS, binned_effect, build_mapping,
                         group_breakdown, group_shap, shap_sample, top_reasons)
from src.features import ID_COL, TARGET_COL
from src.metrics import report
from src.models import RANDOM_STATE, cv_predict, make_lgbm, to_lgbm

PROCESSED = Path("../data/processed")
MODELS = Path("../models")
pd.set_option("display.width", 140)

df = load_features(PROCESSED / "train_features.parquet")
dev, _ = split_dev_holdout(df, PROCESSED / "holdout_ids.csv")  # hold-out отбрасываем
y = dev[TARGET_COL]
X_all = dev.drop(columns=[TARGET_COL, ID_COL])
cfg = json.loads((MODELS / "lgbm_best_params.json").read_text())


def make_model():
    return make_lgbm(n_estimators=cfg["n_estimators"], learning_rate=cfg["learning_rate"], **cfg["params"])

## 1. Новая раскладка групп (на прежней модели 07, выборка та же)

In [ ]:
model = joblib.load(MODELS / "lgbm_dev.joblib")
Xs, ys, sv, base, proba = shap_sample(model, to_lgbm(X_all), y, n=20_000, seed=42)

mapping = build_mapping(Xs.columns)
mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=Xs.columns)
share = mean_abs / mean_abs.sum() * 100
by_group = share.groupby(mapping.reindex(share.index).values).sum().sort_values(ascending=False)
by_group.index = [f"{g}: {GROUPS[g]}" for g in by_group.index]
print(by_group.round(1))
print("\nНе сопоставлено:", int((mapping == "other").sum()), "признаков")

## 2. Состав новых и спорных групп

In [ ]:
for g in ["prev_terms", "prev_history", "prev_payments", "prev_closing", "prev_amounts", "prev_other",
          "loan_burden", "credit_vs_goods", "bureau_active", "bureau_volume", "contacts"]:
    print(f"\n=== {g}: {GROUPS[g]} ===")
    print(group_breakdown(sv, Xs, g, top=6))

## 3. Признаки с неоднозначным направлением
Средний вклад SHAP по квантилям значения. Если вклад меняет знак по ходу значений,
зависимость немонотонная и формулировка причины должна учитывать, какая сторона плохая.

In [ ]:
for f in ["CREDIT_TERM", "AMT_GOODS_PRICE", "BUREAU_DAYS_CREDIT_ENDDATE_MAX",
          "PREV_DAYS_LAST_DUE_MAX", "PREV_DAYS_TERMINATION_MAX"]:
    print(f"\n--- {f} ---")
    print(binned_effect(sv, Xs, f, q=5))

In [ ]:
# Категориальные: какие значения повышают и какие снижают риск
for f in ["ORGANIZATION_TYPE", "OCCUPATION_TYPE", "NAME_INCOME_TYPE"]:
    t = binned_effect(sv, Xs, f)
    t = t[t["count"] >= 200]
    print(f"\n--- {f}: выше всего риск ---")
    print(t.head(5))
    print(f"--- {f}: ниже всего риск ---")
    print(t.tail(5))

## 4. Цена принципа «только объяснимые признаки»
Сравниваем на тех же 5 фолдах и с теми же гиперпараметрами (число деревьев фиксировано):
- full: все признаки (OOF из 06);
- без защищённых: убираем только группу protected (результат 08, пересчитываем ради попарных разниц);
- только объяснимые: убираем все группы из EXCLUDED_GROUPS (защищённые, регион, окружение, технические).

In [ ]:
mapping_all = build_mapping(X_all.columns)
drop_protected = [c for c in X_all.columns if mapping_all[c] == "protected"]
drop_excluded = [c for c in X_all.columns if mapping_all[c] in EXCLUDED_GROUPS]

print("Убираем в варианте 'без защищённых':", len(drop_protected))
print("Убираем в варианте 'только объяснимые':", len(drop_excluded))
print(mapping_all[drop_excluded].value_counts().to_string())

In [ ]:
# опорные фолды: OOF настроенной модели, сохранённый в 06
oof_saved = pd.read_parquet(PROCESSED / "oof_dev.parquet")
assert (oof_saved[ID_COL].values == dev[ID_COL].values).all(), "порядок строк dev изменился"
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
fold_full = np.array([
    roc_auc_score(y.iloc[va], oof_saved["LightGBM_tuned"].iloc[va]) for _, va in skf.split(X_all, y)
])

variants = {
    "без защищённых": X_all.drop(columns=drop_protected),
    "только объяснимые": X_all.drop(columns=drop_excluded),
}
results = {}
for name, Xv in variants.items():
    oof, fa = cv_predict(make_model, to_lgbm(Xv), y)
    results[name] = (oof, fa)
    print(f"{name}: AUC {fa.mean():.4f} ± {fa.std():.4f} | признаков: {Xv.shape[1]}")

In [ ]:
rows = [{**report("full", y, oof_saved["LightGBM_tuned"].values),
         "diff vs full (mean)": 0.0, "min": 0.0, "max": 0.0,
         "diff vs без защищённых": np.nan}]
for name, (oof, fa) in results.items():
    d = fa - fold_full
    d2 = fa - results["без защищённых"][1]
    rows.append({**report(name, y, oof), "diff vs full (mean)": round(d.mean(), 4),
                 "min": round(d.min(), 4), "max": round(d.max(), 4),
                 "diff vs без защищённых": round(d2.mean(), 4)})
pd.DataFrame(rows).set_index("model")

## 5. Причины отказа у модели «только объяснимые»
Обучаем её на всём dev, считаем SHAP и смотрим причины.
Скрытых групп в этой модели нет: топ-3 видят всё, что влияет на решение.

In [ ]:
X_expl = to_lgbm(variants["только объяснимые"])
model_e = make_model().fit(X_expl, y)
joblib.dump(model_e, MODELS / "lgbm_explainable.joblib")
(MODELS / "features_explainable.json").write_text(json.dumps(list(X_expl.columns), ensure_ascii=False, indent=1))

Xs_e, ys_e, sv_e, base_e, proba_e = shap_sample(model_e, X_expl, y, n=20_000, seed=42)
grouped = group_shap(sv_e, Xs_e.columns)
reasons = top_reasons(grouped, k=3, min_shap=0.0)

In [ ]:
order = np.argsort(proba_e)
examples = {"высокий риск": order[-1], "риск около 20%": order[np.searchsorted(proba_e[order], 0.20)],
            "средний риск": order[len(order) // 2]}
for label, i in examples.items():
    print(f"--- {label}: вероятность дефолта {proba_e[i]:.1%} | факт: {ys_e.iloc[i]} ---")
    for text, val in reasons[i]:
        print(f"  {text}  (вклад {val:+.2f})")
    print()

In [ ]:
# Частота причин у верхних 10% по риску и покрытие
risky = np.where(proba_e >= np.quantile(proba_e, 0.9))[0]
counts = pd.Series([t for i in risky for t, _ in reasons[i]]).value_counts()
print("Частота причин у верхних 10% по риску, %:")
print((counts / len(risky) * 100).round(1))

pos_total = np.clip(grouped.values, 0, None).sum(axis=1)
shown = np.array([sum(v for _, v in reasons[i]) for i in range(len(reasons))])
print(f"\nДоля положительного вклада, покрытая топ-3 (верхний дециль): {np.mean(shown[risky] / pos_total[risky]):.1%}")

In [ ]:
# Для выбора минимального порога вклада: насколько велики вклады 1-й, 2-й и 3-й причины у рискованных клиентов
for k in range(3):
    vals = np.array([reasons[i][k][1] for i in risky if len(reasons[i]) > k])
    print(f"Причина №{k + 1}: квантили вклада 10% / 50% / 90%: {np.quantile(vals, [0.1, 0.5, 0.9]).round(2)}")

Что смотрим:
1. цена принципа в AUC и стабильна ли она по фолдам (`min`, `max`);
2. насколько велика доля, которую даёт регион, окружение и технические признаки по отдельности
   (разница между «без защищённых» и «только объяснимые»);
3. стали ли причины конкретнее после разбиения групп, исчезла ли расплывчатость;
4. какой порог вклада отсекает слабые причины, но не оставляет клиентов вовсе без причин.